# Qdrant: one collection, many books

Storing records with both vectors, per-book tenancy, and switching index versions without a gap.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
from qdrant_client import QdrantClient
from marginalia.index import store
from marginalia.index.store import Scope
from tests.conftest import FakeDense, FakeSparse, synthetic_book
from marginalia.ingest.pipeline import build_records, summary_records, embed
client = QdrantClient(':memory:')          # the same API as the Docker Qdrant
dense, sparse = FakeDense(), FakeSparse()
store.ensure_collection(client, 'books', dense.dim)
chapters, summaries = synthetic_book()
records = build_records(chapters, 'demo-book', 1, 'single_curly') + summary_records(chapters, summaries, 'demo-book', 1)
d, s = embed(records, dense, sparse, log=lambda *_: None)
store.upsert(client, 'books', records, d, s)
print(store.count(client, 'books', book_id='demo-book'), 'points')

## Reading back by filters

In [ ]:
scope = Scope('demo-book', 1)
for p in store.fetch(client, 'books', scope, kind='summary'):
    print(p['label'], '|', p['text'])

## Hybrid search, scoped to one book and version

In [ ]:
hits = store.hybrid_search(client, 'books', scope, dense.embed_query('Lazarus Gospel'), sparse.embed_query('Lazarus Gospel'),
                           kind='chunk', limit=3)
for h in hits:
    print(round(h['score'], 3), h['label'], h['text'][:70])

## A new index version, then the switch
Re-indexing writes version 2 next to version 1; the book's profile then points at 2 and version 1 is deleted. A search never sees a half-built index.

In [ ]:
v2 = build_records(chapters, 'demo-book', 2, 'single_curly')
d2, s2 = embed(v2, dense, sparse, log=lambda *_: None)
store.upsert(client, 'books', v2, d2, s2)
print('both versions:', store.count(client, 'books', book_id='demo-book'))
store.delete_book(client, 'books', 'demo-book', keep_version=2)
print('after the switch:', store.count(client, 'books', book_id='demo-book'),
      '| version 1 left:', store.count(client, 'books', Scope('demo-book', 1)))

## The real collection (needs the stack)

In [ ]:
from marginalia.config import Settings
real = QdrantClient(url=Settings().qdrant_url)
for c in real.get_collections().collections:
    print(c.name, real.count(c.name).count)